# PROBLEM (House Price Prediction):

By Diego Fiz

_______________________________________________________________________________________________________________________________________________________

# House Price Prediction - Regression and Inverse Optimization Problem

## Context

A real estate analyst needs to predict house prices and find optimal configurations that maximize desirable features (area, bedrooms, bathrooms, stories) while minimizing cost within a given budget. The dataset contains 545 houses with 12 features including area, number of rooms, and amenities.

---

## Decision Variables

- **x₁**: Area (m²)
- **x₂**: Bedrooms (1-6)
- **x₃**: Bathrooms (1-4)
- **x₄**: Stories (1-4)
- **x₅, ..., x₁₂**: Binary variables (mainroad, guestroom, basement, hotwaterheating, airconditioning, prefarea) and categorical (furnishing status)
- **y**: House price (target variable)

---

## Parameters

| Feature | Min Value | Max Value | Mean | Description |
|---------|-----------|-----------|------|-------------|
| **Area** | 1,650 m² | 16,200 m² | 5,151 m² | Living area |
| **Bedrooms** | 1 | 6 | ~3 | Number of bedrooms |
| **Bathrooms** | 1 | 4 | ~2 | Number of bathrooms |
| **Stories** | 1 | 4 | ~2 | Number of floors |
| **Price** | ~1M € | ~13M € | 4,766,729 € | Target variable |

**Dataset**: 545 houses, 12 variables

**Binary Features**: mainroad, guestroom, basement, hotwaterheating, airconditioning, prefarea (encoded as 0/1)

**Categorical**: Furnishing status (0=unfurnished, 1=semi-furnished, 2=furnished)

---

## Objective Function

### Phase 1: Prediction Model

Train regression models to predict house prices:

$$\text{Price} = w_0 + w_1 \cdot \text{area} + w_2 \cdot \text{bedrooms} + \ldots + w_{12} \cdot \text{furnishing}$$

**Method 1 - Least Squares:**

Minimize:

$$\min_w ||Xw - y||^2$$

Solution:

$$w = (X^T X)^{-1} X^T y$$

**Method 2 - Ridge Regression:**

Minimize:

$$\min_w ||Xw - y||^2 + \lambda||w||^2$$

Solution:

$$w = (X^T X + \lambda I)^{-1} X^T y$$

**Method 3 - Gradient Descent:**

Iteration:

$$w_{k+1} = w_k - \alpha \nabla f(w_k)$$

Gradient:

$$\nabla f(w) = \frac{2}{n} X^T(Xw - y)$$

---

### Phase 2: Inverse Optimization (SLSQP)

**Maximize** value score:

$$\max \left( \text{area} + 500 \times \text{bedrooms} + 500 \times \text{bathrooms} + 300 \times \text{stories} \right)$$

Transformed for minimization:

$$\min \left( -\text{area} - 500 \times \text{bedrooms} - 500 \times \text{bathrooms} - 300 \times \text{stories} \right)$$

---

## Constraints

### 1. Budget Constraint:

$$\text{Predicted Price} \leq \text{Budget}$$

where:

$$\text{Predicted Price} = \left( \frac{x - \mu_X}{\sigma_X} \cdot w \right) \sigma_y + \mu_y$$

### 2. Minimum Area Constraint:

$$\text{area} \geq 3{,}000 \text{ m}^2$$

### 3. Feature Bounds:

$$1{,}650 \leq \text{area} \leq 16{,}200$$

$$1 \leq \text{bedrooms} \leq 6$$

$$1 \leq \text{bathrooms} \leq 4$$

$$1 \leq \text{stories} \leq 4$$

### 4. Binary Constraints:

$$x_i \in \{0, 1\} \quad \text{for } i = 5, 6, 7, 8, 9, 10$$

### 5. Categorical Constraint:

$$\text{furnishing} \in \{0, 1, 2\}$$

### 6. Non-negativity:

$$x_i \geq 0 \quad \forall i$$

---

#### Solution:

In [ ]:
"""
================================================================================
OPTIMIZACIÓN DE PRECIOS DE VIVIENDAS - REGRESIÓN LINEAL PURA
================================================================================
OBJETIVO:
- Predecir precios mediante regresión lineal
- Identificar configuraciones que maximicen área+estancias minimizando precio
================================================================================
"""

import pandas as pd
import numpy as np
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================================
# PASO 1: CARGA Y PREPROCESAMIENTO
# ============================================================================

df = pd.read_csv('Housing_Price_Data.csv')

# Codificar variables categóricas
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating',
               'airconditioning', 'prefarea']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

furnishing_map = {'unfurnished': 0, 'semi-furnished': 1, 'furnished': 2}
df['furnishingstatus'] = df['furnishingstatus'].map(furnishing_map)

X = df.drop('price', axis=1).values
y = df['price'].values
feature_names = df.drop('price', axis=1).columns.tolist()

n_samples, n_features = X.shape

print(f"\nDataset:")
print(f"   Houses: {n_samples}")
print(f"   Variables: {n_features}")
print(f"   Average Price: €{y.mean():,.0f}")
print(f"   Average Area: {df['area'].mean():.0f} m²")


Dataset:
   Houses: 545
   Variables: 12
   Average Price: €4,766,729
   Average Area: 5151 m²


In [ ]:
# ============================================================================
# PASO 2: STANDARDIZATION
# ============================================================================

X_mean = np.mean(X, axis=0)
X_std = np.std(X, axis=0)
X_scaled = (X - X_mean) / X_std

y_mean = np.mean(y)
y_std = np.std(y)
y_scaled = (y - y_mean) / y_std

X_with_bias = np.column_stack([np.ones(n_samples), X_scaled])

print(f"\nStandardization completed")
print(f"   Media X: {np.mean(X_scaled):.10f} ≈ 0")
print(f"   Std X: {np.mean(np.std(X_scaled, axis=0)):.10f} ≈ 1")


Standardization completed
   Media X: -0.0000000000 ≈ 0
   Std X: 1.0000000000 ≈ 1


In [ ]:
# ============================================================================
# PASO 3: LEAST SQUARES
# ============================================================================

print("\n" + "="*90)
print("METHOD 1 - LEAST SQUARES")
print("="*90)
print("\nSolution: w = (X^T X)^(-1) X^T y")

XtX = X_with_bias.T @ X_with_bias
Xty = X_with_bias.T @ y_scaled
w_ls = np.linalg.solve(XtX, Xty)

y_pred_ls = X_with_bias @ w_ls
r2_ls = 1 - np.sum((y_scaled - y_pred_ls)**2) / np.sum((y_scaled - np.mean(y_scaled))**2)
mse_ls = np.mean((y_scaled - y_pred_ls)**2)

y_pred_ls_orig = y_pred_ls * y_std + y_mean
rmse_ls = np.sqrt(np.mean((y - y_pred_ls_orig)**2))

print(f"\nResults:")
print(f"   R²: {r2_ls:.4f} ({r2_ls*100:.2f}%)")
print(f"   RMSE: €{rmse_ls:,.2f}")


METHOD 1 - LEAST SQUARES

Solution: w = (X^T X)^(-1) X^T y

Results:
   R²: 0.6801 (68.01%)
   RMSE: €1,056,995.06


In [ ]:
# ============================================================================
# PASO 4: RIDGE REGRESSION
# ============================================================================

print("\n" + "="*90)
print("METHOD 2 - RIDGE REGRESSION")
print("="*90)
print("\nSolution: w = (X^T X + λI)^(-1) X^T y")

lambdas = [1000, 1000, 1000]
best_ridge_r2 = -np.inf
best_ridge_lambda = None
best_ridge_w = None

for lam in lambdas:
    XtX_ridge = XtX + lam * np.eye(n_features + 1)
    w_ridge = np.linalg.solve(XtX_ridge, Xty)


    y_pred_ridge = X_with_bias @ w_ridge
    r2_ridge = 1 - np.sum((y_scaled - y_pred_ridge)**2) / np.sum((y_scaled - np.mean(y_scaled))**2)

    if r2_ridge > best_ridge_r2:
        best_ridge_r2 = r2_ridge
        best_ridge_lambda = lam
        best_ridge_w = w_ridge

y_pred_ridge = X_with_bias @ best_ridge_w
y_pred_ridge_orig = y_pred_ridge * y_std + y_mean
rmse_ridge = np.sqrt(np.mean((y - y_pred_ridge_orig)**2))

print(f"\nResults (λ = {best_ridge_lambda}):")
print(f"   R²: {best_ridge_r2:.4f} ({best_ridge_r2*100:.2f}%)")
print(f"   RMSE: €{rmse_ridge:,.2f}")


METHOD 2 - RIDGE REGRESSION

Solution: w = (X^T X + λI)^(-1) X^T y

Results (λ = 1000):
   R²: 0.5441 (54.41%)
   RMSE: €1,261,719.91


In [ ]:
# ============================================================================
# PASO 5: GRADIENT DESCENT
# ============================================================================

print("\n" + "="*90)
print("METHOD 3 - GRADIENT DESCENT")
print("="*90)
print("\nIteration: w^(k+1) = w^k - α∇f(w^k)")

def gradient_descent(X, y, learning_rate=0.01, max_iter=5000, tol=1e-6):
    n, p = X.shape
    w = np.zeros(p)

    for iteration in range(max_iter):
        grad = (2.0 / n) * (X.T @ (X @ w - y))
        w_new = w - learning_rate * grad

        if np.linalg.norm(w_new - w) < tol:
            break
        w = w_new

    return w, iteration + 1

w_gd, iterations = gradient_descent(X_with_bias, y_scaled, learning_rate=0.01)

y_pred_gd = X_with_bias @ w_gd
r2_gd = 1 - np.sum((y_scaled - y_pred_gd)**2) / np.sum((y_scaled - np.mean(y_scaled))**2)
y_pred_gd_orig = y_pred_gd * y_std + y_mean
rmse_gd = np.sqrt(np.mean((y - y_pred_gd_orig)**2))

print(f"\nResults (α=0.01):")
print(f"   R²: {r2_gd:.4f} ({r2_gd*100:.2f}%)")
print(f"   RMSE: €{rmse_gd:,.2f}")
print(f"   Iteraciones: {iterations}")


METHOD 3 - GRADIENT DESCENT

Iteration: w^(k+1) = w^k - α∇f(w^k)

Results (α=0.01):
   R²: 0.6801 (68.01%)
   RMSE: €1,056,995.07
   Iteraciones: 817


In [ ]:
# ============================================================================
# COMPARACIÓN
# ============================================================================

print("\n" + "="*90)
print("MODEL COMPARISON")
print("="*90)

comparison = pd.DataFrame({
    'Model': ['Least Squares', 'Ridge', 'Gradient Descent'],
    'R²': [r2_ls, best_ridge_r2, r2_gd],
    'RMSE (€)': [rmse_ls, rmse_ridge, rmse_gd]
})
comparison = comparison.sort_values('R²', ascending=False)
print("\n" + comparison.to_string(index=False))

best_model = comparison.iloc[0]['Model']
if best_model == 'Least Squares':
    best_w = w_ls
elif best_model == 'Ridge':
    best_w = best_ridge_w
else:
    best_w = w_gd

print(f"\nBest Model: {best_model}")


MODEL COMPARISON

           Model       R²     RMSE (€)
   Least Squares 0.680069 1.056995e+06
Gradient Descent 0.680069 1.056995e+06
           Ridge 0.544135 1.261720e+06

Best Model: Least Squares


In [ ]:
# ============================================================================
# PASO 6: OPTIMIZACIÓN INVERSA CON SLSQP
# ============================================================================

print("\n" + "="*90)
print("OPTIMIZATION - MAXIMIZE AREA + ROOMS, MINIMIZE PRICE") #ROOMS = ESTANCIAS
print("="*90)

def optimize_house(w, X_mean, X_std, y_mean, y_std, target_price, area_min=3000):
    """
    Maximiza: área + bedrooms + bathrooms + stories
    Sujeto a: precio ≤ target_price, área ≥ area_min
    """

    def objective(x):
        x_scaled = (x - X_mean) / X_std
        x_with_bias = np.concatenate([[1], x_scaled])
        y_pred_scaled = x_with_bias @ w
        price_pred = y_pred_scaled * y_std + y_mean

        # Valor: área + peso por estancias
        value = x[0] + 500*x[1] + 500*x[2] + 300*x[3]
        price_penalty = max(0, (price_pred - target_price)/1e6) * 10000

        return -(value - price_penalty)

    def constraint_price(x):
        x_scaled = (x - X_mean) / X_std
        x_with_bias = np.concatenate([[1], x_scaled])
        y_pred_scaled = x_with_bias @ w
        price_pred = y_pred_scaled * y_std + y_mean
        return target_price - price_pred

    def constraint_area(x):
        return x[0] - area_min

    bounds = [
        (1650, 16200), (1, 6), (1, 4), (1, 4),
        (0, 1), (0, 1), (0, 1), (0, 1),
        (0, 1), (0, 3), (0, 1), (0, 2)
    ]

    constraints = [
        {'type': 'ineq', 'fun': constraint_price},
        {'type': 'ineq', 'fun': constraint_area}
    ]

    x0 = np.array([3000, 2, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0], dtype=float)

    result = minimize(objective, x0, method='SLSQP', bounds=bounds,
                     constraints=constraints, options={'maxiter': 1000})

    return result

# Optimizar para diferentes presupuestos
print("\nBUDGET OPTIMIZATIONS:\n")
budgets = [3000000, 4000000, 5000000, 6000000, 7000000]

for budget in budgets:
    result = optimize_house(best_w, X_mean, X_std, y_mean, y_std, budget)

    if result.success:
        x_opt = result.x

        # Calcular precio
        x_scaled = (x_opt - X_mean) / X_std
        x_with_bias = np.concatenate([[1], x_scaled])
        y_pred_scaled = x_with_bias @ best_w
        price_pred = y_pred_scaled * y_std + y_mean

        area = int(x_opt[0])
        bedrooms = max(1, int(round(x_opt[1])))
        bathrooms = max(1, int(round(x_opt[2])))
        stories = max(1, int(round(x_opt[3])))
        total_rooms = bedrooms + bathrooms + stories

        print(f"{'='*80}")
        print(f"BUDGET: €{budget:,.0f}")
        print(f"\n Established Price: €{price_pred:,.0f}")
        print(f"Area: {area:,} m² | Rooms: {total_rooms}")
        print(f"  • Bedrooms: {bedrooms} | Bathrooms: {bathrooms} | Floors: {stories}")
        print(f"  • Efficiency: {area/price_pred*1000:.2f} m²/€1000\n")


OPTIMIZATION - MAXIMIZE AREA + ROOMS, MINIMIZE PRICE

BUDGET OPTIMIZATIONS:

BUDGET: €4,000,000

 Established Price: €4,000,000
Area: 8,890 m² | Rooms: 8
  • Bedrooms: 6 | Bathrooms: 1 | Floors: 1
  • Efficiency: 2.22 m²/€1000

BUDGET: €5,000,000

 Established Price: €5,000,000
Area: 12,990 m² | Rooms: 8
  • Bedrooms: 6 | Bathrooms: 1 | Floors: 1
  • Efficiency: 2.60 m²/€1000

BUDGET: €6,000,000

 Established Price: €6,000,000
Area: 16,199 m² | Rooms: 8
  • Bedrooms: 6 | Bathrooms: 1 | Floors: 1
  • Efficiency: 2.70 m²/€1000

BUDGET: €7,000,000

 Established Price: €7,000,000
Area: 16,200 m² | Rooms: 11
  • Bedrooms: 6 | Bathrooms: 1 | Floors: 4
  • Efficiency: 2.31 m²/€1000



##### Graphics:

In [ ]:
"""
================================================================================
GRÁFICOS DE SENSIBILIDAD Y VISUALIZACIÓN - SOLO CONCEPTOS DEL PDF
================================================================================
Teoría utilizada:
- Lesson 2, págs 23-37: Sensitivity Analysis (Shadow Prices)
- Lesson 1, pág 55: Gradient Descent Convergence
- Lesson 1, págs 49-51: Residual Analysis (Least Squares)
- Lesson 2, págs 30-32: Multi-objective (Pareto Front)
- Lesson 4, pág 36: Convexity visualization
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# PREPARACIÓN DE DATOS
# ============================================================================
df = pd.read_csv('Housing_Price_Data.csv')

# Codificar variables
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating',
               'airconditioning', 'prefarea']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

furnishing_map = {'unfurnished': 0, 'semi-furnished': 1, 'furnished': 2}
df['furnishingstatus'] = df['furnishingstatus'].map(furnishing_map)

X = df.drop('price', axis=1).values
y = df['price'].values
feature_names = df.drop('price', axis=1).columns.tolist()
n_samples, n_features = X.shape

# Normalización (Numerical Conditioning - Lesson 4, pág 93)
X_mean = np.mean(X, axis=0)
X_std = np.std(X, axis=0)
X_scaled = (X - X_mean) / X_std

y_mean = np.mean(y)
y_std = np.std(y)
y_scaled = (y - y_mean) / y_std

X_with_bias = np.column_stack([np.ones(n_samples), X_scaled])

# Least Squares
XtX = X_with_bias.T @ X_with_bias
Xty = X_with_bias.T @ y_scaled
w_ls = np.linalg.solve(XtX, Xty)
y_pred_ls = X_with_bias @ w_ls
y_pred_orig = y_pred_ls * y_std + y_mean

In [ ]:
# ============================================================================
# GRÁFICO 1: CONVERGENCIA DE GRADIENT DESCENT
# ============================================================================
print("\nGenerando: Convergencia de Gradient Descent...")

def gradient_descent_tracked(X, y, learning_rate=0.01, max_iter=2000):
    """Gradient Descent con tracking"""
    n, p = X.shape
    w = np.zeros(p)
    mse_history = []

    for iteration in range(max_iter):
        pred = X @ w
        mse = np.mean((y - pred)**2)
        mse_history.append(mse)

        grad = X.T @ (pred - y) / n
        w = w - learning_rate * grad

        if iteration > 20 and abs(mse_history[-1] - mse_history[-2]) < 1e-9:
            break

    return w, mse_history

w_gd, mse_history = gradient_descent_tracked(X_with_bias, y_scaled,
                                              learning_rate=0.01, max_iter=2000)

plt.figure(figsize=(10, 6))
plt.plot(mse_history, linewidth=2, color='#2E86AB')
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('MSE (Mean Squared Error)', fontsize=12)
plt.title('Gradient Descent Convergence', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.axhline(y=mse_history[-1], color='red', linestyle='--',
            label=f'Final MSE: {mse_history[-1]:.4f}')
plt.legend()
plt.tight_layout()
plt.savefig('01_gradient_descent_convergence.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"   ✓ Converged in {len(mse_history)} iterations")
print(f"   ✓ MSE reduction: {(1 - mse_history[-1]/mse_history[0])*100:.2f}%")


Generando: Convergencia de Gradient Descent...


NameError: name 'X_with_bias' is not defined

![01_gradient_descent_convergence.png](attachment:01_gradient_descent_convergence.png)

In [ ]:
# ============================================================================
# GRÁFICO 2: RESIDUAL PLOT
# ============================================================================
print("\nGenerando: Residual Analysis (Least Squares)...")

residuals = y - y_pred_orig

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2a. Residuals vs Predicted
axes[0].scatter(y_pred_orig, residuals, alpha=0.5, s=30, color='#A23B72')
axes[0].axhline(y=0, color='black', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Price (€)', fontsize=11)
axes[0].set_ylabel('Residuals (€)', fontsize=11)
axes[0].set_title('Residuals vs Predicted Values', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# 2b. Histogram of Residuals
axes[1].hist(residuals, bins=30, color='#F18F01', alpha=0.7, edgecolor='black')
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residuals (€)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Distribution of Residuals', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Residual Analysis - Least Squares',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('02_residual_analysis.png', dpi=300, bbox_inches='tight')
plt.close()

print(f"   ✓ Mean residual: €{np.mean(residuals):,.2f}")
print(f"   ✓ Std residual: €{np.std(residuals):,.2f}")


Generando: Residual Analysis (Least Squares)...
   ✓ Mean residual: €-0.00
   ✓ Std residual: €1,056,995.06


![02_residual_analysis.png](attachment:02_residual_analysis.png)

In [ ]:
# ============================================================================
# GRÁFICO 3: SENSITIVITY ANALYSIS - SHADOW PRICES
# ============================================================================
print("\nGenerando: Sensitivity Analysis (Shadow Prices)...")

# Calcular importancia de coeficientes (shadow prices análogo)
coef_importance = np.abs(w_ls[1:])  # Excluir bias
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': coef_importance
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(coef_df)))
plt.barh(coef_df['Feature'], coef_df['Importance'], color=colors, edgecolor='black')
plt.xlabel('|Coefficient| (Standardized)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Feature Importance / Shadow Prices',
          fontsize=14, fontweight='bold')
plt.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('03_shadow_prices_sensitivity.png', dpi=300, bbox_inches='tight')
plt.close()

print(f"   ✓ Most important: {coef_df.iloc[-1]['Feature']}")
print(f"   ✓ Least important: {coef_df.iloc[0]['Feature']}")


Generando: Sensitivity Analysis (Shadow Prices)...
   ✓ Most important: area
   ✓ Least important: bedrooms


![03_shadow_prices_sensitivity.png](attachment:03_shadow_prices_sensitivity.png)

In [ ]:
# ============================================================================
# GRÁFICO 4: PREDICTED VS ACTUAL (Validation)
# ============================================================================
print("\nGenerando: Predicted vs Actual Values...")

r2 = 1 - np.sum(residuals**2) / np.sum((y - np.mean(y))**2)

plt.figure(figsize=(10, 8))
plt.scatter(y, y_pred_orig, alpha=0.5, s=40, color='#6A4C93', edgecolor='black', linewidth=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=3, label='Perfect Prediction')
plt.xlabel('Actual Price (€)', fontsize=12)
plt.ylabel('Predicted Price (€)', fontsize=12)
plt.title(f'Predicted vs Actual Prices (R² = {r2:.4f})', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('04_predicted_vs_actual.png', dpi=300, bbox_inches='tight')
plt.close()

print(f"   ✓ R² Score: {r2:.4f}")
print(f"   ✓ RMSE: €{np.sqrt(np.mean(residuals**2)):,.2f}")


Generando: Predicted vs Actual Values...
   ✓ R² Score: 0.6801
   ✓ RMSE: €1,056,995.06


![04_predicted_vs_actual.png](attachment:04_predicted_vs_actual.png)

In [ ]:
# ============================================================================
# GRÁFICO 5: PARETO FRONT - MULTI-OBJECTIVE
# ============================================================================
print("\nGenerando: Pareto Front (Multi-objective Optimization)...")

# Crear soluciones: área vs precio
areas = df['area'].values
prices_pred = y_pred_orig
total_rooms = df['bedrooms'].values + df['bathrooms'].values

# Normalizar para Pareto
areas_norm = (areas - areas.min()) / (areas.max() - areas.min())
prices_norm = (prices_pred - prices_pred.min()) / (prices_pred.max() - prices_pred.min())

# Calcular eficiencia
efficiency = areas / prices_pred * 1000  # m²/€1000

# Identificar Pareto frontier (simplificado)
pareto_indices = []
for i in range(len(areas)):
    dominated = False
    for j in range(len(areas)):
        if i != j:
            # Mejor si: más área Y menor precio
            if areas[j] >= areas[i] and prices_pred[j] <= prices_pred[i]:
                if areas[j] > areas[i] or prices_pred[j] < prices_pred[i]:
                    dominated = True
                    break
    if not dominated:
        pareto_indices.append(i)

plt.figure(figsize=(12, 8))
plt.scatter(prices_pred, areas, alpha=0.4, s=50, color='gray', label='All Solutions')
plt.scatter(prices_pred[pareto_indices], areas[pareto_indices],
           s=100, color='red', marker='*', edgecolor='black', linewidth=1.5,
           label=f'Pareto Optimal ({len(pareto_indices)} solutions)', zorder=5)
plt.xlabel('Predicted Price (€)', fontsize=12)
plt.ylabel('Area (m²)', fontsize=12)
plt.title('Pareto Front: Maximize Area, Minimize Price',
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('05_pareto_front_multiobjetive.png', dpi=300, bbox_inches='tight')
plt.close()

print(f"   ✓ Pareto optimal solutions: {len(pareto_indices)}")
print(f"   ✓ Best efficiency: {efficiency.max():.2f} m²/€1000")


Generando: Pareto Front (Multi-objective Optimization)...
   ✓ Pareto optimal solutions: 19
   ✓ Best efficiency: 2.56 m²/€1000


![05_pareto_front_multiobjetive.png](attachment:05_pareto_front_multiobjetive.png)

In [ ]:
# ============================================================================
# GRÁFICO 6: SENSITIVITY TO REGULARIZATION (Ridge λ)
# ============================================================================
print("\nGenerando: Sensitivity to Regularization Parameter λ...")

lambdas = np.logspace(-3, 3, 50)
r2_scores = []
coef_norms = []

for lam in lambdas:
    XtX_ridge = XtX + lam * np.eye(n_features + 1)
    w_ridge = np.linalg.solve(XtX_ridge, Xty)
    y_pred_ridge = X_with_bias @ w_ridge
    r2 = 1 - np.sum((y_scaled - y_pred_ridge)**2) / np.sum((y_scaled - np.mean(y_scaled))**2)
    r2_scores.append(r2)
    coef_norms.append(np.linalg.norm(w_ridge[1:]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 6a. R² vs λ
axes[0].semilogx(lambdas, r2_scores, linewidth=2, color='#0077B6', marker='o', markersize=4)
axes[0].set_xlabel('Regularization Parameter λ (log scale)', fontsize=11)
axes[0].set_ylabel('R² Score', fontsize=11)
axes[0].set_title('Model Performance vs Regularization', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=max(r2_scores), color='red', linestyle='--', alpha=0.5)

# 6b. Coefficient Norm vs λ
axes[1].semilogx(lambdas, coef_norms, linewidth=2, color='#F07167', marker='s', markersize=4)
axes[1].set_xlabel('Regularization Parameter λ (log scale)', fontsize=11)
axes[1].set_ylabel('||w|| (L2 Norm of Coefficients)', fontsize=11)
axes[1].set_title('Coefficient Shrinkage vs Regularization', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Sensitivity to Ridge Regularization',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('06_ridge_regularization_sensitivity.png', dpi=300, bbox_inches='tight')
plt.close()

print(f"   ✓ Best λ: {lambdas[np.argmax(r2_scores)]:.4f}")
print(f"   ✓ Max R²: {max(r2_scores):.4f}")


Generando: Sensitivity to Regularization Parameter λ...
   ✓ Best λ: 0.0010
   ✓ Max R²: 0.6801


![06_ridge_regularization_sensitivity.png](attachment:06_ridge_regularization_sensitivity.png)

In [ ]:
# ============================================================================
# GRÁFICO 7: OBJECTIVE FUNCTION LANDSCAPE (2D slice)
# ============================================================================
print("\nGenerando: Objective Function Landscape (Convexity)...")

# Tomar 2 coeficientes más importantes para visualizar
top_2_idx = np.argsort(coef_importance)[-2:]
w1_idx, w2_idx = top_2_idx[0] + 1, top_2_idx[1] + 1  # +1 por bias

# Crear grid alrededor del óptimo
w_opt = w_ls.copy()
w1_range = np.linspace(w_opt[w1_idx] - 0.5, w_opt[w1_idx] + 0.5, 50)
w2_range = np.linspace(w_opt[w2_idx] - 0.5, w_opt[w2_idx] + 0.5, 50)
W1, W2 = np.meshgrid(w1_range, w2_range)

# Calcular MSE para cada punto
MSE = np.zeros_like(W1)
for i in range(len(w1_range)):
    for j in range(len(w2_range)):
        w_temp = w_opt.copy()
        w_temp[w1_idx] = W1[j, i]
        w_temp[w2_idx] = W2[j, i]
        y_pred_temp = X_with_bias @ w_temp
        MSE[j, i] = np.mean((y_scaled - y_pred_temp)**2)

plt.figure(figsize=(12, 9))
contour = plt.contour(W1, W2, MSE, levels=20, cmap='viridis')
plt.clabel(contour, inline=True, fontsize=8)
plt.contourf(W1, W2, MSE, levels=20, cmap='viridis', alpha=0.6)
plt.colorbar(label='MSE')
plt.plot(w_opt[w1_idx], w_opt[w2_idx], 'r*', markersize=20,
         label=f'Optimal Point (MSE={mse_history[-1]:.4f})', markeredgecolor='white', markeredgewidth=2)
plt.xlabel(f'w[{feature_names[top_2_idx[0]]}]', fontsize=12)
plt.ylabel(f'w[{feature_names[top_2_idx[1]]}]', fontsize=12)
plt.title('Objective Function Landscape - Convexity (Lesson 4)',
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('07_objective_landscape_convexity.png', dpi=300, bbox_inches='tight')
plt.close()

print(f"   ✓ Visualized dimensions: {feature_names[top_2_idx[0]]}, {feature_names[top_2_idx[1]]}")
print(f"   ✓ Optimal MSE: {mse_history[-1]:.6f}")


Generando: Objective Function Landscape (Convexity)...
   ✓ Visualized dimensions: bathrooms, area
   ✓ Optimal MSE: 0.319931


![07_objective_landscape_convexity.png](attachment:07_objective_landscape_convexity.png)